In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from torch import nn
from transformers import BertModel

# This is the blueprint for your model. It must be run first.
class SentimentClassifier(nn.Module):

  def __init__(self, n_classes):
    super(SentimentClassifier, self).__init__()
    self.bert = BertModel.from_pretrained('bert-base-cased')
    self.drop = nn.Dropout(p=0.2)
    self.out = nn.Linear(self.bert.config.hidden_size, n_classes)

  def forward(self, input_ids, attention_mask):
    _, pooled_output = self.bert(
      input_ids=input_ids,
      attention_mask=attention_mask,
      return_dict=False
    )
    output = self.drop(pooled_output)
    return self.out(output)

In [ ]:
# First, ensure you've installed gradio
!pip install gradio -q

import gradio as gr
import torch
import torch.nn.functional as F
from transformers import BertTokenizer

# -------- 1. SETUP (Same as your training notebook) --------

# Make sure to run the cell defining your model class first!
# class SentimentClassifier(nn.Module):
#   ... (this class must be defined in your notebook)

# Define device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Define class names and tokenizer name
PRE_TRAINED_MODEL_NAME = 'bert-base-cased'
class_names = ['positive','negative']

# Load the tokenizer
tokenizer = BertTokenizer.from_pretrained(PRE_TRAINED_MODEL_NAME)

# Load your best model
best_model_path = "/content/drive/MyDrive/ImdbDataset/bert_checkpoints/best_model_state.bin"
model = SentimentClassifier(len(class_names))
model.load_state_dict(torch.load(best_model_path, map_location=device))
model = model.to(device)
model.eval() # Set model to evaluation mode

# -------- 2. CORRECTED PREDICTION FUNCTION --------

def predict_sentiment(text):
    # This function now correctly tokenizes the input text

    # Use the tokenizer to encode the input text
    encoding = tokenizer.encode_plus(
      text,
      add_special_tokens=True,
      max_length=400, # Use the same MAX_LEN as in training
      return_token_type_ids=False,
      padding='max_length',
      return_attention_mask=True,
      return_tensors='pt',
      truncation=True
    )

    # Move the input tensors to the same device as the model (GPU)
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    # Get model prediction (no gradients needed)
    with torch.no_grad():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

    # Get the predicted class (0 for negative, 1 for positive)
    _, prediction = torch.max(outputs, dim=1)

    # Get the probabilities using softmax
    probs = F.softmax(outputs, dim=1)

    # Return the probabilities for each class as a dictionary
    # Gradio's "Label" component expects this format
    return {class_names[0]: probs[0][0].item(), class_names[1]: probs[0][1].item()}

# -------- 3. GRADIO INTERFACE (Unchanged) --------

# Your interface definition is correct
iface = gr.Interface(
    fn=predict_sentiment,
    inputs=gr.Textbox(lines=5, placeholder="Enter your movie review here..."),
    outputs=gr.Label(num_top_classes=2),
    title="IMDB Movie Review Sentiment Analyzer",
    description="Enter a movie review to see the predicted sentiment (Positive or Negative)."
)

# Launch the app
iface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5d8e1a2dedc0884c24.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
